# Topic 15: Designing a network in PyTorch

CSCI 3320 · Neural Networks III

This notebook is the code from the [Topic 15 lecture note](https://dongchul.kim/csci3320-fall2026/topic/13c), in the same order:
a network picture turned into `nn.Sequential`, the iris 4-16-3 network with its learning
curves, and three network sizes on three datasets. Run the cells from top to bottom, then
change things in the **Try it** cells.

It runs on a CPU; the longest cell (27 small training runs) takes about half a minute.
The printed numbers can differ a little from the note on a different PyTorch version.

## 1. From a picture to code: the 4-16-3 network

- Each set of arrows is one `nn.Linear(in_features, out_features)`.
- Each hidden layer is followed by `nn.ReLU()`; the last layer has none.
- **Rule 1, neighbors match**: one layer's `out_features` is the next one's `in_features`.
- **Rule 2, the ends come from the data**: features in, classes out.

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(4, 16),   # 4 features in, 16 hidden neurons out
    nn.ReLU(),          # bend: after every hidden layer
    nn.Linear(16, 3),   # 16 in (Rule 1), 3 classes out (Rule 2)
)
print(model)

### Counting the weights

`nn.Linear(a, b)` holds $a \times b + b$ numbers.

In [ ]:
for name, p in model.named_parameters():
    print(name, tuple(p.shape), p.numel())
print("total:", sum(p.numel() for p in model.parameters()))

### Checking the shapes before training

In [ ]:
x = torch.zeros(5, 4)       # 5 rows, 4 features
print(model(x).shape)

### When a rule is broken

The first layer gives 8 numbers but the next one expects 16. The `try` lets the notebook
keep going so you can read the message.

In [ ]:
bad = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(16, 3))   # 8, then 16
try:
    bad(torch.zeros(32, 4))
except RuntimeError as e:
    print("RuntimeError:", e)

Read it as: the data arriving is 32 rows by **8** columns, but this layer expects **16**.
The 32 is just the batch size.

**Try it.** Build a 4-32-16-3 network (two hidden layers), print it, and count its weights.

In [ ]:
# Try it: your 4-32-16-3 network

## 2. Look at the data first

Samples, features, classes, the range of the values, and the first rows. These decide the
two ends of the network and tell you whether you need to scale.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_digits

for load in [load_iris, load_wine, load_breast_cancer, load_digits]:
    d = load()
    X, y = d.data, d.target
    print(f"{load.__name__}: {X.shape[0]} samples, {X.shape[1]} features, {len(d.target_names)} classes, "
          f"values from {X.min():g} to {X.max():g}")
    print("    samples per class:", dict(zip(map(str, d.target_names), np.bincount(y).tolist())))

**Try it.** Change `load_wine` to any of the other three and look at its first rows.

In [ ]:
d = load_wine()
df = pd.DataFrame(d.data, columns=d.feature_names)
df["label"] = d.target
df.head()

## 3. Training on iris and drawing the learning curves

The five-step loop from Topic 14. New around it: `StandardScaler` (fit on the training set
only), `Adam`, and a `history` that records loss and accuracy on both sets after every epoch.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

torch.manual_seed(0)
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
scaler = StandardScaler().fit(X_train)                 # learn mean and std from the training set only
X_train = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train, y_test = torch.tensor(y_train), torch.tensor(y_test)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
for epoch in range(200):
    model.train()
    for xb, yb in loader:                              # the same five steps as Topic 14
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
    model.eval()                                       # after every epoch: score both sets
    with torch.no_grad():
        for name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
            out = model(X)
            history[name + "_loss"].append(loss_fn(out, y).item())
            history[name + "_acc"].append((out.argmax(dim=1) == y).float().mean().item())
    if (epoch + 1) in (1, 10, 50, 100, 200):
        print(f"epoch {epoch + 1:3d}  train loss {history['train_loss'][-1]:.3f}"
              f"  test loss {history['test_loss'][-1]:.3f}  test acc {history['test_acc'][-1]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["test_loss"], label="test")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["test_acc"], label="test")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].legend()
plt.show()

Both losses fall together and stay close: a healthy run. The note's run ends at test
accuracy 0.967 (29 of 30 flowers).

**Try it.** Change `range(200)` to `range(500)` and run the cell again. Does the test loss
keep falling?

## 4. Three sizes, built by one function

`hidden` is a list of hidden-layer sizes. `prev` carries Rule 1; `n_in` and `n_out` are Rule 2.

In [ ]:
def make_model(n_in, hidden, n_out):
    """[64, 32] -> Linear(n_in, 64), ReLU, Linear(64, 32), ReLU, Linear(32, n_out)"""
    layers, prev = [], n_in
    for h in hidden:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        prev = h                                         # this layer's output is the next one's input
    layers.append(nn.Linear(prev, n_out))                # the last layer: no ReLU
    return nn.Sequential(*layers)


print(make_model(64, [64, 32], 10))

## 5. The experiment: 3 sizes x 3 datasets x 3 seeds

`prepare` splits and scales; `train` is the iris loop and returns the test curve.
Each combination is trained with three seeds and the mean test accuracy is printed.
About half a minute on a CPU.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_wine, load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def prepare(X, y, seed):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    scaler = StandardScaler().fit(X_tr)
    f = lambda a: torch.tensor(a, dtype=torch.float32)
    return f(scaler.transform(X_tr)), f(scaler.transform(X_te)), torch.tensor(y_tr), torch.tensor(y_te)


def train(model, X_tr, y_tr, X_te, y_te, epochs=200, seed=0):
    torch.manual_seed(seed)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    test_loss, test_acc = [], []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            out = model(X_te)
            test_loss.append(loss_fn(out, y_te).item())
            test_acc.append((out.argmax(dim=1) == y_te).float().mean().item())
    return test_loss, test_acc


SIZES = {"Small": [16], "Medium": [64, 32], "Large": [256, 128, 64]}
DATA = {"Wine": load_wine, "Breast cancer": load_breast_cancer, "Digits": load_digits}

for data_name, load in DATA.items():
    X, y = load(return_X_y=True)
    n_in, n_out = X.shape[1], len(np.unique(y))         # sizes come from the data
    for size_name, hidden in SIZES.items():
        accs = []
        for seed in (0, 1, 2):                           # three different splits and starts
            X_tr, X_te, y_tr, y_te = prepare(X, y, seed)
            torch.manual_seed(seed)
            model = make_model(n_in, hidden, n_out)
            test_loss, test_acc = train(model, X_tr, y_tr, X_te, y_te, seed=seed)
            accs.append(test_acc[-1])
        n_params = sum(p.numel() for p in model.parameters())
        print(f"{data_name:14s} {size_name:7s} {n_params:7,d} params   test accuracy {np.mean(accs):.3f}")

## 6. Breast cancer: watch the test loss

Seed 0, the three sizes on one plot. The bigger networks reach their lowest test loss
within a few epochs and then get worse: they are memorizing the 455 training rows.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = prepare(X, y, 0)
plt.figure(figsize=(7, 3.8))
for size_name, hidden in SIZES.items():
    torch.manual_seed(0)
    model = make_model(X.shape[1], hidden, 2)
    test_loss, test_acc = train(model, X_tr, y_tr, X_te, y_te, seed=0)
    best = int(np.argmin(test_loss))
    print(f"{size_name:7s} lowest test loss {test_loss[best]:.3f} at epoch {best + 1:3d}, "
          f"at epoch 200 {test_loss[-1]:.3f}")
    plt.plot(test_loss, label=f"{size_name} {hidden}")
plt.xlabel("epoch"); plt.ylabel("test loss"); plt.ylim(0, 0.6); plt.legend()
plt.title("Breast cancer, seed 0: test loss")
plt.show()

## 7. Try it: your own design

Pick a dataset and a list of hidden sizes, and read the curves:

- train accuracy stays low: too small, make it **bigger**;
- train accuracy reaches 1.0 but the test loss climbs: memorizing, make it **smaller**;
- both still improving at the end: **train longer**.

In [ ]:
MY_DATA = load_digits          # load_wine, load_breast_cancer or load_digits
MY_HIDDEN = [128, 64]          # your hidden layers
SEED = 0

X, y = MY_DATA(return_X_y=True)
X_tr, X_te, y_tr, y_te = prepare(X, y, SEED)
torch.manual_seed(SEED)
model = make_model(X.shape[1], MY_HIDDEN, len(np.unique(y)))
print(model)
print("weights:", sum(p.numel() for p in model.parameters()))
test_loss, test_acc = train(model, X_tr, y_tr, X_te, y_te, seed=SEED)
print(f"final test accuracy {test_acc[-1]:.3f}, lowest test loss {min(test_loss):.3f} "
      f"at epoch {int(np.argmin(test_loss)) + 1}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(test_loss); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("test loss")
ax[1].plot(test_acc); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("test accuracy")
plt.show()

## What to take away

The ends of a network come from the data; the middle is your choice. Start small, plot the
curves, and grow the network only when the data has more to learn and enough rows to learn
it from. Back to the [lecture note](https://dongchul.kim/csci3320-fall2026/topic/13c).